# Phase 03 — Descriptor-based Gradient Boosting

This notebook evaluates **one frozen cohort at a time** using RDKit descriptors, Morgan fingerprints, MACCS keys and optional 2D Mordred descriptors. It always reads the original MongoDB documents through the shared cleaning loader. Neither the manifest files nor the historical notebooks are changed.

The five stored outer folds provide the out-of-fold (OOF) evaluation. Their saved inner validation IDs are read without resplitting; each inner training set is reconstructed as the complement of its validation IDs. Column filtering, missing-value imputation, mutual information and permutation importance are all fitted **inside** each inner training fold.

The MD-common cohort is used here only to define a shared set of samples. This first pipeline does **not** use MD features as predictors.

## 1. Configuration

Choose a cohort and whether to add Mordred descriptors. Set `RUN_FULL_BENCHMARK = True` only after checking the manifest and runtime budget. The original hyperparameter grid has 54 candidates per inner fold, so the full 5 × 5 nested CV is computationally demanding, especially for PolyInfo. First run the disabled-training validation cells; then test execution separately with a reduced grid, never use outer-test performance to choose hyperparameters.

In [10]:
import hashlib
import sys
import tempfile
from datetime import datetime
from pathlib import Path

import pandas as pd
from IPython.display import display
from pymongo import MongoClient
from dotenv import load_dotenv

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'runlog.py').exists() and (p / 'src/final_comparison/datasets.py').exists()), None)
if ROOT is None:
    raise RuntimeError('Run the notebook from the polymer repository')
sys.path.insert(0, str(ROOT))

from src.final_comparison.datasets import COLLECTIONS, DB_NAME, MD_COLUMNS, load_cohort
from src.final_comparison.splits import load_manifest, fold_ids
from src.final_comparison.gbr import PARAM_GRID, benchmark_gbr, validate_frozen_splits
from runlog import RunLogger

load_dotenv(ROOT / '.env')

COHORT = 'Tg_bicerano'  # Tg_md_common, Tg_bicerano, Tg_polyinfo, or one of four other properties
USE_MORDRED = True
RUN_FULL_BENCHMARK = True
SEED = 28

COHORTS = {
    'Tg_md_common': ('bicerano', 'Tg', True),
    'Tg_bicerano': ('bicerano', 'Tg', False),
    'Tg_polyinfo': ('polyinfo', 'Tg', False),
    'density_extended': ('extended', 'density', False),
    'Tc_extended': ('extended', 'Tc', False),
    'glass_cte_bicerano': ('bicerano', 'glass_cte', False),
    'rubber_cte_bicerano': ('bicerano', 'rubber_cte', False),
}
dataset, target, require_md_coverage = COHORTS[COHORT]
MD_DIR = ROOT / 'data/md_features'
MANIFEST = ROOT / 'data/splits/v1' / f'{COHORT}.json'
if not MANIFEST.is_file():
    raise FileNotFoundError(f'Frozen split manifest is missing: {MANIFEST}')
MANIFEST_SHA256 = hashlib.sha256(MANIFEST.read_bytes()).hexdigest()
TARGET_UNITS = {
    'Tg': 'K', 'density': 'unit not verified for Extended',
    'Tc': 'unit not verified', 'glass_cte': '10^-6 K^-1',
    'rubber_cte': '10^-6 K^-1',
}


## 2. Load and validate the frozen cohort

`load_cohort` performs the established target-specific cleaning directly from MongoDB. `load_manifest` verifies the saved split metadata and source fingerprint. `validate_frozen_splits` checks every saved outer and inner partition (outer fold IDs are 1–5).

In [11]:
md_sources = (
    {name: MD_DIR / name for name in MD_COLUMNS}
    if require_md_coverage else None
)
with MongoClient(__import__('os').environ['MONGO_URI']) as mongo:
    frame, exclusions = load_cohort(
        dataset, target=target, md_dir=MD_DIR if require_md_coverage else None,
        client=mongo,
    )

manifest = load_manifest(MANIFEST, frame=frame, source_paths=md_sources)
ids, verified_folds = validate_frozen_splits(frame, manifest, fold_ids=fold_ids)
assert len(verified_folds) == manifest['n_outer_folds'] == 5
assert len(ids) == len(frame)

print(f'{COHORT}: {len(frame)} samples; {len(exclusions)} exclusions; manifest verified')
print(f'Descriptors: RDKit, Morgan, MACCS' + (' and Mordred' if USE_MORDRED else ''))
print(f'Nested CV: 5 outer × 5 stored inner folds; full training enabled: {RUN_FULL_BENCHMARK}')
print(f'Target unit: {TARGET_UNITS[target]}')


Tg_bicerano: 312 samples; 3 exclusions; manifest verified
Descriptors: RDKit, Morgan, MACCS and Mordred
Nested CV: 5 outer × 5 stored inner folds; full training enabled: True
Target unit: K


## 3. Frozen nested cross-validation

The final model in each outer fold sees only that fold's training data. Within the inner search, the entire preprocessing and feature-selection pipeline is fitted again on each inner training subset. The held-out outer fold is used only for the final evaluation.

Outputs are OOF predictions indexed by `sample_id`, fold-by-fold metrics and hyperparameters, selected features and their stability, and elapsed training times. `MAE/σ` uses the standard deviation of the entire selected cohort as its denominator; this is an evaluation normalization, not a model input.

In [12]:
if RUN_FULL_BENCHMARK:
    log = RunLogger(
        config={
            'db': DB_NAME, 'collection': COLLECTIONS[dataset], 'target': target,
            'cohort': COHORT, 'manifest': str(MANIFEST.relative_to(ROOT)),
            'manifest_sha256': MANIFEST_SHA256,
            'use_md': False, 'md_only': False, 'use_mordred': USE_MORDRED,
            'target_unit': TARGET_UNITS[target],
            'random_state': SEED, 'param_grid': PARAM_GRID,
            'selection': 'training-only MI and OOF permutation importance',
        },
        tag=f'phase03_{COHORT}_{datetime.now():%H%M%S_%f}',
    )
    log.dataset(db=DB_NAME, collection=COLLECTIONS[dataset], n_excluded=len(exclusions))
    with tempfile.TemporaryDirectory(prefix='gbr_cv_cache_') as cache:
        oof, folds, features, metrics = benchmark_gbr(
            frame, manifest, fold_ids=fold_ids, use_mordred=USE_MORDRED,
            log=log, param_grid=PARAM_GRID, seed=SEED, cache_dir=cache,
        )
    assert hashlib.sha256(MANIFEST.read_bytes()).hexdigest() == MANIFEST_SHA256
    log.finish()
    display(pd.DataFrame([metrics]), folds)
    display(oof.head(), features.head())
else:
    print('Training is disabled.')


[runlog] C:\Users\Vale\OneDrive\Desktop\uni\tirocinio\polymer_prediction\runs\11-21_1605_biceranoPolymers_nomd_mrd_phase03_Tg_bicerano_160530_080329


C:\Users\Vale\OneDrive\Desktop\uni\tirocinio\tgEnv310\lib\site-packages\numpy\core\fromnumeric.py:88: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


# Run `11-21_1605_biceranoPolymers_nomd_mrd_phase03_Tg_bicerano_160530_080329`

- **inizio**: 2026-11-21T16:05:30  |  **fine**: 2026-11-21T16:48:23  |  **durata**: 42.9 min
- **commit**: `95c76a1`
- **tag**: phase03_Tg_bicerano_160530_080329

## Dati

- **sorgente**: MongoDB `PolymerPrediction.biceranoPolymers`
- **target**: Tg
- **polimeri usati**: 312
- **descrittori del monomero**: RDKit + Morgan + MACCS + Mordred
- **range Tg**: 130 - 685 K (media 365, std 119)

## Feature MD

- **usate**: NO (`USE_MD = False`)
- il run e' la sola pipeline RDKit: un solo braccio (`base`), niente confronto base/full.

## Configurazione

- **cohort**: Tg_bicerano
- **manifest**: data\splits\v1\Tg_bicerano.json
- **manifest_sha256**: ce842f7b240339cb5c6195817d75a8f75b81f9d2f7c7c755c862778ed0a9d6fa
- **md_only**: False
- **use_mordred**: True
- **target_unit**: K
- **random_state**: 28
- **param_grid**:
  - **gbr__n_estimators**:
    - 100, 200, 400
  - **gbr__learning_rate**:
    - 0.01, 0.05, 0.1
  -

C:\Users\Vale\OneDrive\Desktop\uni\tirocinio\polymer_prediction\runlog.py:377: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([pd.read_csv(index_path), df], ignore_index=True)


,n_samples,n_outer,n_inner,MAE,R2,MAE_over_sigma,fold_MAE_mean,fold_MAE_std,selection_jaccard,fit_seconds
0,312,5,5,32.301554,0.851494,0.270543,32.313104,4.736086,0.498945,2509.178658


,fold,n_train,n_test,n_selected,inner_MAE,MAE,R2,MAE_over_sigma,fit_seconds,best_params
0,1,249,63,15,32.890777,28.854575,0.910000,0.241673,485.255024,"{'gbr__learning_rate': 0.05, 'gbr__max_depth':..."
1,2,249,63,15,33.249265,32.168126,0.869720,0.269426,496.729621,"{'gbr__learning_rate': 0.05, 'gbr__max_depth':..."
2,3,250,62,22,31.382039,38.414612,0.683208,0.321744,507.134420,"{'gbr__learning_rate': 0.1, 'gbr__max_depth': ..."
3,4,250,62,15,33.405434,26.740262,0.898076,0.223965,520.924610,"{'gbr__learning_rate': 0.1, 'gbr__max_depth': ..."
4,5,250,62,14,32.332415,35.387944,0.811349,0.296394,499.134983,"{'gbr__learning_rate': 0.1, 'gbr__max_depth': ..."


,sample_id,fold,y_true,y_pred
0,698ddceeb7014ee539b3abe6,2,130.0,243.693200
1,698ddceeb7014ee539b3abe7,3,152.0,182.874732
2,698ddceeb7014ee539b3abe8,3,171.0,247.500474
3,698ddceeb7014ee539b3abe9,4,173.0,161.368565
4,698ddceeb7014ee539b3abea,3,187.0,384.770041


,fold,feature
0,1,mrd_Xch-7d
1,1,mrd_AATS1dv
2,1,mrd_BalabanJ
3,1,mrd_AATS1d
4,1,mrd_AATS2d
